In [1]:
import spacy
from spacy.tokens import DocBin
import pandas as pd
import numpy as np

df_marking = pd.read_csv('df_with_marking_final.csv')

/Users/ekaterinastepura/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
import re

def get_full_text(row):
    return f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"

male_forms = ["потерпевший", "потерпевшему", "потерпевшего", "потерпевшем"]
female_forms = ["потерпевшая", "потерпевшей", "потерпевшую"]

male_pattern = r"\b(" + "|".join(male_forms) + r")\b"
female_pattern = r"\b(" + "|".join(female_forms) + r")\b"

train_data_victim = []

for idx, row in df_marking.iterrows():
    text = get_full_text(row)
    text_lower = text.lower()
    gender = row["gender_victim"]

    if gender == "неизвестно" or pd.isnull(gender):
        continue

    found = False

    if gender == "мужчина":
        match = re.search(male_pattern, text_lower)
        if match:
            start, end = match.span()
            train_data_victim.append((text, {"entities": [(start, end, "GENDER_VICTIM")]}))
            found = True

    elif gender == "женщина":
        match = re.search(female_pattern, text_lower)
        if match:
            start, end = match.span()
            train_data_victim.append((text, {"entities": [(start, end, "GENDER_VICTIM")]}))
            found = True

    if not found:
        print(f"Не удалось найти ключевое слово для пола потерпевшего '{gender}' в тексте id={row['id']}")

print(f"TRAIN_DATA_VICTIM готово: {len(train_data_victim)} примеров")

Не удалось найти ключевое слово для пола потерпевшего '['женщина', 'женщина']' в тексте id=41579
Не удалось найти ключевое слово для пола потерпевшего '['женщина', 'женщина']' в тексте id=87680
Не удалось найти ключевое слово для пола потерпевшего '['мужчина', 'мужчина']' в тексте id=109467
Не удалось найти ключевое слово для пола потерпевшего '['мужчина', 'мужчина']' в тексте id=113143
Не удалось найти ключевое слово для пола потерпевшего '['женщина', 'мужчина']' в тексте id=110287
Не удалось найти ключевое слово для пола потерпевшего '['мужчина', 'женщина']' в тексте id=81127
TRAIN_DATA_VICTIM готово: 94 примеров


In [3]:
import spacy
from spacy.training.example import Example
from spacy.util import minibatch
import random
import warnings

nlp = spacy.blank("ru")

ner = nlp.add_pipe("ner")

ner.add_label("GENDER_VICTIM")

examples = []
for text, annotations in train_data_victim:
    doc = nlp.make_doc(text)
    examples.append(Example.from_dict(doc, annotations))

n_iter = 30
optimizer = nlp.begin_training()

for i in range(n_iter):
    random.shuffle(examples)
    losses = {}
    batches = minibatch(examples, size=4)

    for batch in batches:
        nlp.update(batch, losses=losses)

    print(f"Итерация {i+1}/{n_iter} — потери: {losses}")

nlp.to_disk("ner_gender_victim_model")
print("Модель сохранена в папке 'ner_gender_victim_model'")

Итерация 1/30 — потери: {'ner': 109389.78601590046}
Итерация 2/30 — потери: {'ner': 1936.5000809812975}
Итерация 3/30 — потери: {'ner': 177.59075662081713}
Итерация 4/30 — потери: {'ner': 129.3104863419223}
Итерация 5/30 — потери: {'ner': 106.09808960833588}
Итерация 6/30 — потери: {'ner': 79.92039395039586}
Итерация 7/30 — потери: {'ner': 56.70807654395548}
Итерация 8/30 — потери: {'ner': 64.81222219923119}
Итерация 9/30 — потери: {'ner': 55.33089928964364}
Итерация 10/30 — потери: {'ner': 44.08358617873666}
Итерация 11/30 — потери: {'ner': 40.90754760072882}
Итерация 12/30 — потери: {'ner': 36.650963665840465}
Итерация 13/30 — потери: {'ner': 32.87751964784472}
Итерация 14/30 — потери: {'ner': 28.90052584635441}
Итерация 15/30 — потери: {'ner': 36.04919913705832}
Итерация 16/30 — потери: {'ner': 21.42000207908653}
Итерация 17/30 — потери: {'ner': 23.059522485704044}
Итерация 18/30 — потери: {'ner': 18.324096929262165}
Итерация 19/30 — потери: {'ner': 16.97095376008663}
Итерация 20/30

In [4]:
import spacy

nlp = spacy.load("ner_gender_victim_model")

def get_full_text(row):
    return f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"

true_labels = []
pred_labels = []

def detect_victim_gender(entity_text):
    word = entity_text.lower()

    masculine_forms = [
        "потерпевший", "потерпевшему", "потерпевшего", "потерпевшем"
    ]
    feminine_forms = [
        "потерпевшая", "потерпевшей", "потерпевшую"
    ]

    if word in masculine_forms:
        return "мужчина"
    elif word in feminine_forms:
        return "женщина"
    else:
        return "неизвестно"

for _, row in df_marking.iterrows():
    true_gender = row["gender_victim"]
    if pd.isnull(true_gender) or true_gender == "неизвестно":
        continue 

    text = get_full_text(row)
    doc = nlp(text)

    predicted_gender = "неизвестно"
    for ent in doc.ents:
        if ent.label_ == "GENDER_VICTIM":
            predicted_gender = detect_victim_gender(ent.text)
            print(ent.text, predicted_gender)
            break

    true_labels.append(true_gender)
    pred_labels.append(predicted_gender)

потерпевшему мужчина
Потерпевший мужчина
Потерпевший мужчина
потерпевшему мужчина
потерпевшего мужчина
потерпевшего мужчина
потерпевшей женщина
потерпевшего мужчина
потерпевшего мужчина
потерпевшему мужчина
потерпевшей женщина
потерпевшего мужчина
Потерпевший мужчина
потерпевшего мужчина
потерпевшему мужчина
потерпевшего мужчина
потерпевшего мужчина
потерпевшей женщина
потерпевшего мужчина
потерпевшего мужчина
потерпевшей женщина
потерпевшей женщина
потерпевшей женщина
потерпевшего мужчина
Потерпевший мужчина
потерпевшему мужчина
потерпевшего мужчина
потерпевшего мужчина
потерпевшему мужчина
потерпевшего мужчина
потерпевшего мужчина
Потерпевший мужчина
потерпевшей женщина
потерпевшего мужчина
потерпевшему мужчина
потерпевшего мужчина
потерпевшего мужчина
потерпевшего мужчина
потерпевшему мужчина
Потерпевший мужчина
потерпевшего мужчина
потерпевший мужчина
Потерпевший мужчина
потерпевшей женщина
Потерпевший мужчина
потерпевшей женщина
потерпевшего мужчина
Потерпевший мужчина
потерпевшег

In [9]:
from sklearn.metrics import accuracy_score, f1_score
accuracy = accuracy_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels, average="weighted")
print(f"\nAccuracy: {accuracy:.2f}")
print(f"F1-score: {f1:.2f}")


Accuracy: 0.89
F1-score: 0.91
